In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 04_kmeans_segmentation - Segmentación RFM
# MAGIC Clustering KMeans y etiquetado premium (2016-2018)

# COMMAND ----------

from pyspark.sql import functions as F
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

GOLD_PATH = "/Volumes/olist/olist_gold/gold/"
METRICS_PATH = "/Volumes/olist/olist_gold/metrics/"

# COMMAND ----------

# Crear volume metrics
try:
    spark.sql("CREATE VOLUME IF NOT EXISTS olist.olist_gold.metrics")
except:
    pass

# COMMAND ----------

# Cargar RFM y convertir a Pandas
rfm_pd = spark.read.format("delta").load(f"{GOLD_PATH}rfm_cutoff_20180930/").toPandas()

print(f"📥 {len(rfm_pd):,} clientes cargados\n")

# COMMAND ----------

# Verificar y limpiar NaNs
print("🔍 Verificando datos...\n")

print(f"NaNs por columna:")
print(rfm_pd[["recency", "frequency", "monetary"]].isnull().sum())
print()

# Eliminar filas con NaN en columnas RFM
rfm_clean = rfm_pd[["customer_id", "recency", "frequency", "monetary"]].dropna()

print(f"✅ Registros válidos: {len(rfm_clean):,} (eliminados: {len(rfm_pd) - len(rfm_clean):,})\n")

# COMMAND ----------

# Escalar RFM
X = rfm_clean[["recency", "frequency", "monetary"]].values
X_scaled = StandardScaler().fit_transform(X)

print("✅ Datos escalados\n")

# COMMAND ----------

# Calcular métricas para k=2..12
print("📊 Calculando métricas k=2..12...\n")

k_range = range(2, 13)
inertias = []
silhouettes = []
davies_bouldin = []
calinski_harabasz = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    
    # Inertia
    inertias.append(km.inertia_)
    
    # Silhouette (sample si > 10k)
    if len(X_scaled) > 10000:
        idx = pd.Series(range(len(X_scaled))).sample(10000, random_state=42).values
        sil = silhouette_score(X_scaled[idx], labels[idx])
    else:
        sil = silhouette_score(X_scaled, labels)
    silhouettes.append(sil)
    
    # Davies-Bouldin (menor es mejor)
    db = davies_bouldin_score(X_scaled, labels)
    davies_bouldin.append(db)
    
    # Calinski-Harabasz (mayor es mejor)
    ch = calinski_harabasz_score(X_scaled, labels)
    calinski_harabasz.append(ch)
    
    print(f"k={k:2d} | inertia={km.inertia_:>8,.0f} | sil={sil:.3f} | DB={db:.3f} | CH={ch:>8,.1f}")

print()

# COMMAND ----------

# Calcular tasa de cambio de inercia (método del codo)
print("📐 Análisis del codo:\n")

inertia_changes = []
for i in range(1, len(inertias)):
    change = abs(inertias[i] - inertias[i-1])
    rate = (change / inertias[i-1]) * 100
    inertia_changes.append(rate)
    print(f"k={i+2} → k={i+3}: cambio={rate:.2f}%")

print()

# COMMAND ----------

# Graficar todas las métricas
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Inertia
axes[0, 0].plot(k_range, inertias, 'bo-', linewidth=2)
axes[0, 0].set_xlabel('k', fontsize=10)
axes[0, 0].set_ylabel('Inertia', fontsize=10)
axes[0, 0].set_title('Método del Codo', fontsize=12, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)

# Silhouette
axes[0, 1].plot(k_range, silhouettes, 'go-', linewidth=2)
axes[0, 1].set_xlabel('k', fontsize=10)
axes[0, 1].set_ylabel('Silhouette Score', fontsize=10)
axes[0, 1].set_title('Índice de Silueta (↑ mejor)', fontsize=12, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].axhline(y=max(silhouettes), color='r', linestyle='--', alpha=0.5, label=f'Máx: {max(silhouettes):.3f}')
axes[0, 1].legend()

# Davies-Bouldin
axes[1, 0].plot(k_range, davies_bouldin, 'ro-', linewidth=2)
axes[1, 0].set_xlabel('k', fontsize=10)
axes[1, 0].set_ylabel('Davies-Bouldin Index', fontsize=10)
axes[1, 0].set_title('Davies-Bouldin (↓ mejor)', fontsize=12, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].axhline(y=min(davies_bouldin), color='r', linestyle='--', alpha=0.5, label=f'Mín: {min(davies_bouldin):.3f}')
axes[1, 0].legend()

# Calinski-Harabasz
axes[1, 1].plot(k_range, calinski_harabasz, 'mo-', linewidth=2)
axes[1, 1].set_xlabel('k', fontsize=10)
axes[1, 1].set_ylabel('Calinski-Harabasz Score', fontsize=10)
axes[1, 1].set_title('Calinski-Harabasz (↑ mejor)', fontsize=12, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].axhline(y=max(calinski_harabasz), color='r', linestyle='--', alpha=0.5, label=f'Máx: {max(calinski_harabasz):,.0f}')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

# COMMAND ----------

# Seleccionar k óptimo (máximo silhouette)
k_opt = k_range[silhouettes.index(max(silhouettes))]

print(f"{'='*60}")
print("🎯 SELECCIÓN DE K ÓPTIMO")
print(f"{'='*60}")
print(f"Máximo Silhouette: k={k_opt} (score={max(silhouettes):.3f})")
print(f"Mínimo Davies-Bouldin: k={k_range[davies_bouldin.index(min(davies_bouldin))]} (score={min(davies_bouldin):.3f})")
print(f"Máximo Calinski-Harabasz: k={k_range[calinski_harabasz.index(max(calinski_harabasz))]} (score={max(calinski_harabasz):,.1f})")
print(f"\n✅ K seleccionado: {k_opt}\n")

# COMMAND ----------

# KMeans final
rfm_clean['cluster'] = KMeans(n_clusters=k_opt, random_state=42, n_init=10).fit_predict(X_scaled)

# Ordenar por monetary promedio
cluster_order = rfm_clean.groupby('cluster')['monetary'].mean().sort_values(ascending=False).index
cluster_map = {old: new+1 for new, old in enumerate(cluster_order)}

rfm_clean['cluster_ordered'] = rfm_clean['cluster'].map(cluster_map)
rfm_clean['is_premium'] = (rfm_clean['cluster_ordered'] == 1).astype(int)

premium_count = rfm_clean['is_premium'].sum()
print(f"✅ Clientes premium: {premium_count:,} ({premium_count/len(rfm_clean)*100:.1f}%)\n")

# COMMAND ----------

# Estadísticas por cluster
print("📊 Perfil de clusters:\n")

cluster_profile = rfm_clean.groupby('cluster_ordered').agg({
    'customer_id': 'count',
    'recency': 'mean',
    'frequency': 'mean',
    'monetary': 'mean'
}).round(2)
cluster_profile.columns = ['count', 'avg_recency', 'avg_frequency', 'avg_monetary']

print(cluster_profile)
print()

# COMMAND ----------

# Guardar segmentados (con mergeSchema para evitar error de schema mismatch)
spark.createDataFrame(rfm_clean) \
    .write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .option("overwriteSchema", "true") \
    .save(f"{GOLD_PATH}customers_segmented_20180930/")

print(f"✅ Guardado: customers_segmented_20180930/\n")

# COMMAND ----------

# Guardar métricas completas
metrics = pd.DataFrame({
    'k': list(k_range),
    'inertia': inertias,
    'silhouette': silhouettes,
    'davies_bouldin': davies_bouldin,
    'calinski_harabasz': calinski_harabasz
})

spark.createDataFrame(metrics) \
    .write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(f"{METRICS_PATH}kmeans/")

print(f"✅ Guardado: metrics/kmeans/\n")

# COMMAND ----------

# Resumen final
print(f"{'='*60}")
print("✅ SEGMENTACIÓN COMPLETADA")
print(f"{'='*60}")
print(f"Total clientes: {len(rfm_clean):,}")
print(f"Clusters: {k_opt}")
print(f"Premium: {premium_count:,} ({premium_count/len(rfm_clean)*100:.1f}%)")
print(f"\nMétricas finales (k={k_opt}):")
print(f"  • Inertia: {inertias[k_opt-2]:,.0f}")
print(f"  • Silhouette: {silhouettes[k_opt-2]:.3f}")
print(f"  • Davies-Bouldin: {davies_bouldin[k_opt-2]:.3f}")
print(f"  • Calinski-Harabasz: {calinski_harabasz[k_opt-2]:,.1f}")
print(f"\nDistribución por cluster:")
print(rfm_clean['cluster_ordered'].value_counts().sort_index())



In [0]:
# COMMAND ----------

# MAGIC %md
# MAGIC ## 📊 Exportar Datos para Dashboard

# COMMAND ----------

print("="*60)
print("💾 EXPORTANDO DATOS PARA DASHBOARD")
print("="*60)
print()

DASHBOARD_PATH = "/Volumes/olist/olist_gold/dashboard/"

# Crear volume dashboard si no existe
try:
    spark.sql("CREATE VOLUME IF NOT EXISTS olist.olist_gold.dashboard")
    print("✅ Volume 'dashboard' verificado/creado")
except Exception as e:
    print(f"⚠️  Volume dashboard: {e}")

print()

# COMMAND ----------

# MAGIC %md
# MAGIC ### 1. Exportar Customers Segmentados

# COMMAND ----------

print("1️⃣ Exportando customers_segmented...")
print()

try:
    # Leer tabla ya guardada
    customers_seg = spark.read.format("delta").load(
        f"{GOLD_PATH}customers_segmented_20180930/"
    )
    
    # Guardar en dashboard
    customers_seg.write.format("delta").mode("overwrite") \
        .option("overwriteSchema", "true") \
        .save(f"{DASHBOARD_PATH}customers_segmented/")
    
    print(f"✅ customers_segmented/ exportado")
    print(f"   • Registros: {customers_seg.count():,}")
    print(f"   • Columnas: {len(customers_seg.columns)}")
    
except Exception as e:
    print(f"❌ Error exportando customers: {e}")

print()

# COMMAND ----------

# MAGIC %md
# MAGIC ### 2. Exportar Métricas de K-Means

# COMMAND ----------

print("2️⃣ Exportando kmeans_metrics.csv...")
print()

try:
    # El DataFrame 'metrics' ya existe del código anterior
    # Contiene: k, inertia, silhouette, davies_bouldin, calinski_harabasz
    
    # Verificar que existe
    if 'metrics' not in locals() and 'metrics' not in globals():
        # Si no existe, recrearlo
        metrics = pd.DataFrame({
            'k': list(k_range),
            'inertia': inertias,
            'silhouette': silhouettes,
            'davies_bouldin': davies_bouldin,
            'calinski_harabasz': calinski_harabasz
        })
    
    print(f"   Métricas calculadas para K = {list(metrics['k'])}")
    print()
    
    # Guardar como CSV usando método temporal
    temp_path = f"{DASHBOARD_PATH}kmeans_temp/"
    
    spark.createDataFrame(metrics).write \
        .format("csv").mode("overwrite").option("header", "true") \
        .save(temp_path)
    
    # Mover archivo CSV
    csv_files = [f for f in dbutils.fs.ls(temp_path) if f.name.endswith('.csv')]
    
    if csv_files:
        dbutils.fs.cp(csv_files[0].path, f"{DASHBOARD_PATH}kmeans_metrics.csv")
        print(f"✅ kmeans_metrics.csv exportado")
        print(f"   • K evaluados: {len(metrics)}")
        print(f"   • K óptimo: {k_opt}")
        print(f"   • Mejor Silhouette: {max(silhouettes):.4f}")
    else:
        print(f"❌ No se encontró archivo CSV en {temp_path}")
    
    # Limpiar temporal
    dbutils.fs.rm(temp_path, True)
    
except Exception as e:
    print(f"❌ Error exportando métricas: {e}")

print()

# COMMAND ----------

# MAGIC %md
# MAGIC ### 3. Verificación

# COMMAND ----------

print("3️⃣ Verificando archivos exportados...")
print()

required_files = [
    "customers_segmented/",
    "kmeans_metrics.csv"
]

all_ok = True

for file in required_files:
    try:
        result = dbutils.fs.ls(f"{DASHBOARD_PATH}{file}")
        
        if file.endswith('/'):
            # Es una carpeta Delta
            print(f"✅ {file}")
            print(f"   • Archivos: {len(result)}")
        else:
            # Es un archivo CSV
            print(f"✅ {file}")
            print(f"   • Tamaño: {result[0].size:,} bytes")
    except Exception as e:
        print(f"❌ {file} - NO ENCONTRADO")
        all_ok = False

print()

if all_ok:
    print("✅ TODOS LOS ARCHIVOS EXPORTADOS CORRECTAMENTE")
else:
    print("⚠️  ALGUNOS ARCHIVOS NO SE EXPORTARON")

print()

# COMMAND ----------

# MAGIC %md
# MAGIC ### 4. Resumen de Exportación

# COMMAND ----------

print("="*60)
print("📋 RESUMEN DE EXPORTACIÓN")
print("="*60)
print()

print(f"📂 Ubicación: {DASHBOARD_PATH}")
print()

print("📦 Archivos exportados:")
print()

print("1. customers_segmented/ (Delta)")
print("   • customer_id")
print("   • recency, frequency, monetary")
print("   • cluster, cluster_ordered")
print("   • is_premium")
print()

print("2. kmeans_metrics.csv")
print("   • k (2-12)")
print("   • inertia")
print("   • silhouette")
print("   • davies_bouldin")
print("   • calinski_harabasz")
print()

print("="*60)
print("✅ EXPORTACIÓN COMPLETADA PARA NOTEBOOK 04")
print("="*60)
print()

print("🎯 Próximo paso:")
print("   Ejecutar notebook 05 y agregar su exportación")
print()



###############################################################
# ═══════════════════════════════════════════════════════════════
# AGREGAR AL FINAL DEL NOTEBOOK 04
# Después de la sección "Resumen de Exportación"
# ═══════════════════════════════════════════════════════════════

# COMMAND ----------

# MAGIC %md
# MAGIC ### 5. Crear Tabla en Unity Catalog (para SQL Warehouse)

# COMMAND ----------

print("="*60)
print("📊 CREANDO TABLA EN UNITY CATALOG")
print("="*60)
print()

# Leer CSV recién exportado
try:
    metrics_df = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .load(f"{DASHBOARD_PATH}kmeans_metrics.csv")
    
    print("✅ CSV leído correctamente")
    print(f"   • Registros: {metrics_df.count()}")
    print()
    
    # Mostrar schema
    print("📋 Schema:")
    metrics_df.printSchema()
    print()
    
    # Guardar como tabla Delta en Unity Catalog
    metrics_df.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("olist.olist_gold.kmeans_metrics")
    
    print("✅ Tabla creada: olist.olist_gold.kmeans_metrics")
    print()
    
    # Verificar
    result = spark.sql("SELECT * FROM olist.olist_gold.kmeans_metrics ORDER BY k")
    print("📊 Contenido de la tabla:")
    result.show()
    
    print()
    print("="*60)
    print("✅ TABLA LISTA PARA APP.PY")
    print("="*60)
    print()
    print("🔗 Para consultar desde app.py:")
    print("   SELECT * FROM olist.olist_gold.kmeans_metrics ORDER BY k")
    print()
    
except Exception as e:
    print(f"❌ Error creando tabla: {e}")
    print()
    print("💡 Verifica:")
    print("   1. Que el CSV se exportó correctamente")
    print("   2. Que tienes permisos en Unity Catalog")
    print("   3. Que el schema olist.olist_gold existe")

# COMMAND ----------

# MAGIC %md
# MAGIC ### 6. También crear tabla para customers_segmented

# COMMAND ----------

print("="*60)
print("👥 CREANDO TABLA customers_segmented")
print("="*60)
print()

try:
    # Leer desde dashboard
    customers_df = spark.read.format("delta") \
        .load(f"{DASHBOARD_PATH}customers_segmented/")
    
    print("✅ Datos leídos correctamente")
    print(f"   • Registros: {customers_df.count():,}")
    print()
    
    # Guardar como tabla
    customers_df.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("olist.olist_gold.customers_segmented")
    
    print("✅ Tabla creada: olist.olist_gold.customers_segmented")
    print()
    
    # Verificar
    result = spark.sql("""
        SELECT 
            cluster_ordered,
            COUNT(*) as count,
            SUM(is_premium) as premium_count
        FROM olist.olist_gold.customers_segmented
        GROUP BY cluster_ordered
        ORDER BY cluster_ordered
    """)
    
    print("📊 Resumen por cluster:")
    result.show()
    
    print()
    print("="*60)
    print("✅ TODAS LAS TABLAS LISTAS")
    print("="*60)
    print()
    
except Exception as e:
    print(f"❌ Error creando tabla: {e}")

# COMMAND ----------

print("="*60)
print("🎉 NOTEBOOK 04 COMPLETADO")
print("="*60)
print()
print("📊 Tablas creadas en Unity Catalog:")
print("   ✅ olist.olist_gold.kmeans_metrics")
print("   ✅ olist.olist_gold.customers_segmented")
print()
print("📂 Archivos en Volume:")
print("   ✅ /Volumes/olist/olist_gold/dashboard/kmeans_metrics.csv")
print("   ✅ /Volumes/olist/olist_gold/dashboard/customers_segmented/")
print()
print("🚀 Ahora puedes:")
print("   1. Ejecutar app.py (usará las tablas de Unity Catalog)")
print("   2. Ejecutar convertir_a_tablas.py (para el resto de datos)")
print()